In [3]:
import requests
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from model import DecoderOnlyTransformer

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

with open("shakespeare.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(text[:500])
print("total chars:", len(text))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
total chars: 1115394


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("vocab_size =", tokenizer.vocab_size)
print("pad_token_id =", tokenizer.pad_token_id)
print("eos_token_id =", tokenizer.eos_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab_size = 50257
pad_token_id = 50256
eos_token_id = 50256


In [ ]:
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

token_ids = tokenizer.encode(text)
print("num tokens:", len(token_ids))
print(token_ids[:30])
print(tokenizer.decode(token_ids[:30]))

Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


num tokens: 338025
[5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248, 461, 11, 2740, 13, 198, 198, 5962, 22307, 25, 198]
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:



In [ ]:
class LMDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        self.token_ids = token_ids
        self.seq_len = seq_len

    def __len__(self):
        return len(self.token_ids) - self.seq_len

    def __getitem__(self, idx):
        chunk = self.token_ids[idx : idx + self.seq_len + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)   # input
        y = torch.tensor(chunk[1:], dtype=torch.long)    # label
        return x, y

In [ ]:
seq_len = 128
batch_size = 16

dataset = LMDataset(token_ids, seq_len=seq_len)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

x, y = next(iter(loader))
print(x.shape)   # (batch, seq_len)
print(y.shape)   # (batch, seq_len)

torch.Size([16, 128])
torch.Size([16, 128])


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DecoderOnlyTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    n_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1,
    max_len=seq_len
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

epochs = 3

model.train()
for epoch in range(epochs):
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, attn_weights = model(x)
        # logits: (B, S, V)
        # y:      (B, S)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}, loss = {avg_loss:.4f}")

Epoch 1, loss = 2.5738
Epoch 2, loss = 1.4340
Epoch 3, loss = 1.1167


In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=1.0, device="cpu"):
    model.eval()

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        if input_ids.size(1) > model.pos_encoding.pe.size(1):
            input_cond = input_ids[:, -model.pos_encoding.pe.size(1):]
        else:
            input_cond = input_ids

        logits, _ = model(input_cond)
        next_token_logits = logits[:, -1, :] / temperature
        probs = nn.functional.softmax(next_token_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_id], dim=1)

    return tokenizer.decode(input_ids[0].tolist())

In [ ]:
print(generate(model, tokenizer, prompt="who is the old man", max_new_tokens=80, device=device))

who is the old man; but by
daughter of hiss: he hath gone before
the princess, and many of brave young
customers, and First, and
deceiving promises of life-s flesh which is more
with the choughs of life; they seemed almost to tear
encounter Tybalt, more strong link asunder.

MERCUTIO:
Thou


In [ ]:
print(generate(model, tokenizer, prompt="KING:", max_new_tokens=80, device=device))

KING:
So shall you have: my hope all;
And there I hope I have none; those that die thou wilt,
That I have not keep him long.

TRANIO:
Thou art the man, despite of all ice:
Of thee, that thou hast braved me; thy wild acts denote
Thy thoughts, the next thy duke isa
